## Inspect a trained DDS-VAE scene encoder

Loads the Stage 2 checkpoint written by `src.training.train_dds_vae`, which contains both stages, and inspects
held-out maps from the training split (seed 101):

1. Encoder pipeline: map $\mathbf{m}$, DDS selection, bottleneck $\mathbf{h}$, Stage 1 and full reconstructions
2. Prior samples $z \sim \mathcal{N}(0, I)$ and latent interpolations
3. Posterior statistics: per-dimension KL, aggregate posterior, and KS distance to $\mathcal{N}(0, 1)$
4. UMAP density of posterior means
5. Latent traversals, interactive latent explorer, and posterior temperature

In [ ]:
import base64
import io
import sys
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import torch
import umap
from IPython.display import display
from ipywidgets import HTML, Button, FloatSlider, HBox, IntSlider, Layout, VBox
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import MaxNLocator
from PIL import Image
from scipy import stats
from torch.utils.data import DataLoader, Subset
from torchvision.utils import make_grid

sys.path.insert(0, "..")  # repository root, notebook runs from notebooks/
from src.datasets.maps import split_maps
from src.models.dds_vae import DDSVAE, gaussian_kl, normalize_maps, reparameterize

# ── Config ───────────────────────────────────────────
@dataclass
class NotebookConfig:
    checkpoint: str = "../checkpoints/dds_vae_dmlab/version_0/stage2/last.ckpt"
    map_dir: str = "../data/dmlab_topdown_maps"
    val_fraction: float = 0.05
    seed: int = 101
    num_examples: int = 6
    num_stats_maps: int = 5000  # training maps encoded for the aggregate posterior
    device: str = "cuda"

# ── Model and data ───────────────────────────────────
cfg = NotebookConfig()
device = torch.device(cfg.device)
model = DDSVAE.from_checkpoint(cfg.checkpoint, device)
hparams = torch.load(cfg.checkpoint, map_location="cpu")["hyper_parameters"]
free_bits = hparams["optim"]["free_bits"]
train_set, val_set = split_maps(cfg.map_dir, cfg.val_fraction, cfg.seed)
rng = np.random.default_rng(cfg.seed)
examples = normalize_maps(torch.stack([val_set[i] for i in range(cfg.num_examples)]).to(device))  # [N, 3, 128, 128]

num_params = sum(p.numel() for p in model.parameters())
print(f"[INFO] DDS-VAE: {num_params / 1e6:.2f}M parameters | d_z = {model.latent_dim} | free bits = {free_bits}")
print(f"[INFO] Maps: {len(train_set):,} train / {len(val_set):,} held-out")

# ── Style ────────────────────────────────────────────
PAPER, INK, SLATE, MIST, HAIRLINE, BLUE = "#fbfbfb", "#191b21", "#626b7a", "#9ba6b7", "#dfe3ea", "#164be6"
DIVERGING = LinearSegmentedColormap.from_list("slate_blue", [SLATE, PAPER, BLUE])
TOPO = LinearSegmentedColormap.from_list("paper_blue_violet", [PAPER, "#dbe3fb", "#8fa8f2", BLUE, "#3b2fa8", "#9443d0"])
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "text.color": INK, "axes.labelcolor": SLATE, "xtick.color": SLATE, "ytick.color": SLATE,
    "axes.edgecolor": HAIRLINE, "axes.linewidth": 0.7, "axes.spines.top": False, "axes.spines.right": False,
    "font.family": "sans-serif", "font.sans-serif": ["Inter", "Arial", "DejaVu Sans"],
    "font.monospace": ["DM Mono", "DejaVu Sans Mono"], "font.size": 10.5, "axes.labelsize": 10,
    "axes.titlesize": 10.5, "axes.titlelocation": "left", "xtick.labelsize": 8.5, "ytick.labelsize": 8.5,
    "xtick.major.size": 0, "ytick.major.size": 0,
})

def to_rgb(x):
    """Maps [N, 3, H, W] in [-1, 1] -> numpy [N, H, W, 3] in [0, 1]."""
    return x.float().add(1).div(2).clamp(0, 1).permute(0, 2, 3, 1).cpu().numpy()

def style_axis(ax):
    """Quiet axis with mono tick numbers and at most five ticks."""
    ax.xaxis.set_major_locator(MaxNLocator(5)); ax.yaxis.set_major_locator(MaxNLocator(5))
    for label in ax.get_xticklabels() + ax.get_yticklabels(): label.set_fontfamily("monospace")

def image_grid(rows, row_labels=(), col_labels=(), size=1.5):
    """Borderless grid of rows of RGB [H, W, 3] maps or [H, W] fields in [-1, 1] (diverging colormap)."""
    fig, axes = plt.subplots(len(rows), len(rows[0]), figsize=(len(rows[0]) * size, len(rows) * size), squeeze=False,
                             gridspec_kw={"wspace": 0.03, "hspace": 0.03})
    for ax, image in zip(axes.flat, [image for row in rows for image in row]):
        ax.imshow(image, interpolation="nearest", **({"cmap": DIVERGING, "vmin": -1, "vmax": 1} if image.ndim == 2 else {}))
        ax.set_xticks([]); ax.set_yticks([]); ax.spines[:].set_visible(False)
    for ax, label in zip(axes[:, 0], row_labels): ax.set_ylabel(label, color=SLATE, labelpad=6)
    for ax, label in zip(axes[0], col_labels): ax.set_title(label, color=INK, pad=6)
    return fig, axes

### Encoder pipeline

Each row follows one held-out map through Stage 1 and the mini-VAE. The bottleneck panel tiles the 16 channels of
$\mathbf{h} \in [-1, 1]^{16 \times 16 \times 16}$; the last column decodes the posterior mean $z = \mu$.

In [ ]:
with torch.inference_mode():
    selected = model.backbone.select(examples)
    stage1_recon, h = model.backbone(examples)  # [N, 3, 128, 128], [N, 16, 16, 16]
    vae_recon, _ = model(examples)

# [N, 16, 16, 16] -> [N, 4*16 + pad, 4*16 + pad], one tile per bottleneck channel
h_tiles = [make_grid(hi[:, None], nrow=4, padding=1, pad_value=0)[0].cpu().numpy() for hi in h.float()]
columns = [to_rgb(examples), to_rgb(selected), h_tiles, to_rgb(stage1_recon), to_rgb(vae_recon)]
image_grid([list(row) for row in zip(*columns)], size=1.7,
           col_labels=["Map m", "DDS selection", "Bottleneck h", "Stage 1", "DDS-VAE (z = μ)"])
mse = [(x - examples).square().mean().item() for x in (stage1_recon, vae_recon)]
print(f"[INFO] Held-out example MSE: Stage 1 {mse[0]:.4f} | DDS-VAE {mse[1]:.4f}")
plt.show()

### Prior samples and interpolation

Scene codes drawn from the prior, and linear paths between posterior means of held-out maps.

In [ ]:
generator = torch.Generator(device).manual_seed(cfg.seed)
with torch.inference_mode():
    samples = to_rgb(model.decode(torch.randn(24, model.latent_dim, generator=generator, device=device)))
image_grid([list(samples[i:i + 8]) for i in range(0, 24, 8)])
plt.show()

# steps = torch.linspace(0, 1, 8, device=device)[:, None]  # [S, 1]
# with torch.inference_mode():
#     mu, _ = model.encode(examples)
#     paths = [to_rgb(model.decode((1 - steps) * mu[a] + steps * mu[b])) for a, b in [(0, 1), (2, 3), (4, 5)]]
# image_grid([list(path) for path in paths], col_labels=[f"t = {t:.2f}" for t in steps[:, 0].tolist()])
# plt.show()

### Posterior statistics

Per-dimension KL of the posterior to the prior, sorted, with the free-bits threshold; the pooled aggregate posterior
$z = \mu + \sigma \epsilon$ against $\mathcal{N}(0, 1)$; and the per-dimension Kolmogorov–Smirnov distance with its
critical value at $\alpha = 0.05$.

In [ ]:
@torch.inference_mode()
def encode_maps(dataset, batch_size=256):
    """Posterior mu and log sigma^2 [N, d_z] for a uint8 map dataset."""
    outputs = [model.encode(normalize_maps(maps.to(device))) for maps in DataLoader(dataset, batch_size=batch_size, num_workers=4)]
    return [torch.cat(parts).float().cpu().numpy() for parts in zip(*outputs)]

mu, logvar = encode_maps(Subset(train_set, rng.choice(len(train_set), cfg.num_stats_maps, replace=False).tolist()))
sigma = np.exp(0.5 * logvar)
z = mu + sigma * rng.standard_normal(mu.shape)
kl = gaussian_kl(torch.from_numpy(mu), torch.from_numpy(logvar)).mean(0).numpy()  # [d_z]
order = np.argsort(kl)[::-1]
active = kl > free_bits
ks = np.array([stats.kstest(z[:, d], "norm").statistic for d in range(model.latent_dim)])
ks_critical = stats.kstwo.ppf(0.95, len(z))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), constrained_layout=True)
dims = np.arange(model.latent_dim)
axes[0].bar(dims, kl[order], width=1.0, color=np.where(active[order], BLUE, MIST), linewidth=0)
axes[0].axhline(free_bits, color=SLATE, lw=0.8, ls=(0, (3, 3)))
axes[0].text(dims[-1], free_bits, "free bits", color=SLATE, ha="right", va="bottom", fontsize=8.5)
axes[0].set(xlabel="Latent dimension (sorted)", ylabel="KL to prior (nats)", title=f"{active.sum()} / {len(kl)} active dimensions")
grid = np.linspace(-4, 4, 400)
axes[1].hist(z.ravel(), bins=120, range=(-4, 4), density=True, color=BLUE, alpha=0.35, linewidth=0)
axes[1].plot(grid, stats.norm.pdf(grid), color=INK, lw=1.2)
axes[1].text(1.15, stats.norm.pdf(1.15), "  N(0, 1)", color=INK, va="bottom", fontsize=8.5)
axes[1].set(xlabel="z (pooled over dimensions)", ylabel="Density", title=f"Aggregate posterior · std {z.std():.2f}")
axes[2].plot(dims, np.sort(ks)[::-1], color=BLUE, lw=1.4)
axes[2].axhline(ks_critical, color=SLATE, lw=0.8, ls=(0, (3, 3)))
axes[2].text(dims[-1], ks_critical, "α = 0.05", color=SLATE, ha="right", va="bottom", fontsize=8.5)
axes[2].set(xlabel="Latent dimension (sorted)", ylabel="KS distance to N(0, 1)",
            title=f"{(ks > ks_critical).sum()} / {len(ks)} dimensions reject N(0, 1)")
for ax in axes: style_axis(ax)
print(f"[INFO] {len(z):,} maps | KL {kl.sum():.1f} nats | mean σ {sigma.mean():.3f} | std of μ {mu.std(0).mean():.3f}")
plt.show()

### UMAP density of posterior means

Gaussian KDE of the 2-D UMAP embedding, drawn as filled iso-density bands with contour lines.

In [ ]:
embedding = umap.UMAP(n_neighbors=50, min_dist=0.3, random_state=cfg.seed).fit_transform(mu)  # [N, 2]

# ── Density on a regular grid ─────────────────────────
pad = 0.08 * np.ptp(embedding, axis=0)
low, high = embedding.min(0) - pad, embedding.max(0) + pad
gx, gy = np.meshgrid(np.linspace(low[0], high[0], 240), np.linspace(low[1], high[1], 240))  # [G, G]
density = stats.gaussian_kde(embedding.T, bw_method=0.12)(np.stack([gx.ravel(), gy.ravel()])).reshape(gx.shape)
density /= density.max()  # bandwidth 0.12 of the data std; Scott's rule merges UMAP clusters

# ── Topographic rendering ─────────────────────────────
levels = np.linspace(0.02, 1.0, 18)  # below 2% of the peak stays paper
fig, ax = plt.subplots(figsize=(6.0, 5.0), constrained_layout=True)
bands = ax.contourf(gx, gy, density, levels=levels, cmap=TOPO)
ax.contour(gx, gy, density, levels=levels[::2], colors=INK, linewidths=0.45, alpha=0.35)
colorbar = fig.colorbar(bands, ax=ax, fraction=0.045, pad=0.02, ticks=[0.25, 0.5, 0.75, 1.0])
colorbar.set_label("Relative density", color=SLATE); colorbar.outline.set_visible(False); colorbar.ax.tick_params(length=0)
for label in colorbar.ax.get_yticklabels(): label.set_fontfamily("monospace")
ax.set(xlabel="UMAP 1", ylabel="UMAP 2", xticks=[], yticks=[], title=f"{len(mu):,} training maps")
ax.spines[:].set_visible(False)
plt.show()

### Latent traversals

What each high-KL dimension controls. Every column is one latent dimension $z_d$ (header: index and KL to the prior);
the first column is the held-out map and its reconstruction at $z = \mu$. The two image rows decode the same map with
$z_d$ moved to $\mu_d - 3$ and $\mu_d + 3$ prior standard deviations, all other dimensions fixed. The bottom row is the
absolute pixel change between the two decodes: bright regions are where that dimension edits the scene.

In [ ]:
def effect_figure(rows, row_labels, col_labels, heat_row, cell=1.6):
    """Grid of RGB cells (None = empty); the row `heat_row` holds [H, W] non-negative fields on one shared colour scale."""
    heat_max = max(f.max() for f in rows[heat_row] if f is not None)
    fig, axes = plt.subplots(len(rows), len(rows[0]), figsize=(len(rows[0]) * cell, len(rows) * cell + 0.5), squeeze=False,
                             gridspec_kw={"wspace": 0.04, "hspace": 0.04})
    for r, row in enumerate(rows):
        for c, image in enumerate(row):
            ax = axes[r, c]; ax.set_xticks([]); ax.set_yticks([]); ax.spines[:].set_visible(False)
            if image is None: continue
            ax.imshow(image, **({"cmap": TOPO, "vmin": 0, "vmax": heat_max} if r == heat_row else {}))
    for ax, label in zip(axes[:, 0], row_labels): ax.set_ylabel(label, color=SLATE, labelpad=6)
    for ax, label in zip(axes[0], col_labels): ax.set_title(label, color=INK, pad=6, fontsize=9)
    return fig

traversal_map, top_dims, reach = 0, order[:6].tolist(), 3.0  # reach in prior standard deviations
with torch.inference_mode():
    base, _ = model.encode(normalize_maps(val_set[traversal_map][None].to(device)))  # [1, d_z]
    ends = []
    for d in top_dims:
        z = base.repeat(2, 1); z[:, d] += torch.tensor([-reach, reach], device=device)  # [2, d_z]
        ends.append(model.decode(z))  # [2, 3, H, W]: (-reach, +reach)
    base_rgb = to_rgb(model.decode(base))[0]
minus, plus = [to_rgb(e[:1])[0] for e in ends], [to_rgb(e[1:])[0] for e in ends]
change = [np.abs(a - b).mean(-1) for a, b in zip(minus, plus)]  # [H, W], mean over RGB
source = to_rgb(normalize_maps(val_set[traversal_map][None]))[0]
effect_figure([[source, *minus], [base_rgb, *plus], [None, *change]],
              [f"z_d = μ_d − {reach:g}", f"z_d = μ_d + {reach:g}", "|change|"],
              ["held-out map / z = μ"] + [f"z[{d}]  KL {kl[d]:.2f}" for d in top_dims], heat_row=2)
plt.show()

### Interactive latent explorer

Starts from the posterior mean of a held-out map (left) and offsets the eight dimensions with the highest KL; the
decoded map (right) updates as the sliders move. In VS Code, `jupyter.widgetScriptSources` must allow a CDN for the
widget controls to load.

In [ ]:
val_mu, _ = encode_maps(val_set)
explore_dims = order[:8].tolist()

def to_png(x, size=288):
    """One map [3, H, W] in [-1, 1] -> inline <img> tag at display size (HTML widgets load in every frontend)."""
    image = Image.fromarray((to_rgb(x[None])[0] * 255).astype(np.uint8)).resize((size, size), Image.NEAREST)
    buffer = io.BytesIO(); image.save(buffer, format="PNG")
    return f'<img src="data:image/png;base64,{base64.b64encode(buffer.getvalue()).decode()}" width="{size}" height="{size}">'

source_view, decoded_view = HTML(), HTML()
index = IntSlider(0, min=0, max=len(val_set) - 1, description="map", layout=Layout(width="320px"))
sliders = [FloatSlider(0.0, min=-3, max=3, step=0.05, description=f"Δz[{d}]", layout=Layout(width="320px")) for d in explore_dims]
reset = Button(description="Reset offsets", layout=Layout(width="320px"))

@torch.inference_mode()
def refresh(_=None):
    z = torch.from_numpy(val_mu[index.value]).to(device)
    z[explore_dims] += torch.tensor([slider.value for slider in sliders], device=device)
    source_view.value = to_png(normalize_maps(val_set[index.value][None])[0])
    decoded_view.value = to_png(model.decode(z[None])[0])

def reset_offsets(_):
    for slider in sliders: slider.value = 0.0

for widget in [index, *sliders]: widget.observe(refresh, "value")
reset.on_click(reset_offsets)
refresh()
display(HBox([source_view, decoded_view, VBox([index, *sliders, reset])]))

### Posterior temperature

How much the decoded map depends on posterior noise. Each column draws $z = \mu + T\sigma\epsilon$ for one held-out map
at temperature $T$ ($T = 0$ is the posterior mean). The top row is a single sample; the bottom row is the per-pixel
standard deviation across 16 samples, so bright regions are the parts of the map the posterior leaves uncertain.

In [ ]:
temperatures, num_draws = [0.0, 0.5, 1.0, 2.0], 16
with torch.inference_mode():
    mu, logvar = model.encode(examples[:1].expand(num_draws, -1, -1, -1))  # [K, d_z], K posterior draws of one map
    draws = [model.decode(reparameterize(mu, logvar, t)) for t in temperatures]  # T x [K, 3, H, W]
sample_row = [to_rgb(d[:1])[0] for d in draws]
spread_row = [d.float().std(0).mean(0).cpu().numpy() for d in draws]  # [H, W], std over draws then RGB
effect_figure([sample_row, spread_row], ["one sample", f"pixel std, {num_draws} draws"],
              [f"T = {t:g}" for t in temperatures], heat_row=1, cell=2.0)
plt.show()